# 01 — GoEmotions: Exploratory Data Analysis

Understand the data before touching a model. We'll look at class balance, text lengths,
and vocabulary patterns per emotion, then define our 12-group taxonomy and save clean splits.

In [ ]:
import os
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from datasets import load_dataset

warnings.filterwarnings("ignore")
plt.rcParams.update({
    "figure.dpi": 110,
    "figure.facecolor": "white",
    "axes.spines.top": False,
    "axes.spines.right": False,
})
%matplotlib inline

DATA_DIR   = Path("data")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
(DATA_DIR / "processed").mkdir(parents=True, exist_ok=True)

In [ ]:
# ~50 MB from HuggingFace Hub on first run, then cached locally
ds = load_dataset("google-research-datasets/go_emotions", "simplified")
print(ds)

In [ ]:
label_names = ds["train"].features["labels"].feature.names
print(f"{len(label_names)} fine-grained labels:")
for i, name in enumerate(label_names):
    print(f"  {i:2d}: {name}")

## Macro-Emotion Taxonomy

We collapse the 27 fine-grained labels into 12 macro-groups that have
meaningfully different musical identities for Spotify mapping.

In [ ]:
# Maps each fine-grained label index to one of our 12 macro-emotions.
# Index order matches label_names above.
MACRO_MAP = {
    0:  "admiration",   # admiration
    1:  "joy",          # amusement
    2:  "anger",        # anger
    3:  "anger",        # annoyance
    4:  "joy",          # approval
    5:  "love",         # caring
    6:  "curiosity",    # confusion
    7:  "curiosity",    # curiosity
    8:  "love",         # desire
    9:  "sadness",      # disappointment
    10: "anger",        # disapproval
    11: "disgust",      # disgust
    12: "disgust",      # embarrassment
    13: "excitement",   # excitement
    14: "fear",         # fear
    15: "gratitude",    # gratitude
    16: "sadness",      # grief
    17: "joy",          # joy
    18: "love",         # love
    19: "fear",         # nervousness
    20: "optimism",     # optimism
    21: "admiration",   # pride
    22: "curiosity",    # realization
    23: "gratitude",    # relief
    24: "sadness",      # remorse
    25: "sadness",      # sadness
    26: "excitement",   # surprise
    27: "neutral",      # neutral
}

MACRO_LABELS = sorted(set(MACRO_MAP.values()))
print(f"{len(MACRO_LABELS)} macro-emotions: {MACRO_LABELS}")

In [ ]:
def to_dataframe(split: str) -> pd.DataFrame:
    rows = []
    for ex in ds[split]:
        if not ex["labels"]:
            continue
        # multi-label examples: we take the first label — covers ~95% of cases cleanly
        rows.append({"text": ex["text"], "label": MACRO_MAP[ex["labels"][0]]})
    return pd.DataFrame(rows)

train_df = to_dataframe("train")
val_df   = to_dataframe("validation")
test_df  = to_dataframe("test")

print(f"Train: {len(train_df):,}  |  Val: {len(val_df):,}  |  Test: {len(test_df):,}")
train_df.head()

In [ ]:
counts = train_df["label"].value_counts()

fig, ax = plt.subplots(figsize=(11, 4))
counts.plot(kind="bar", ax=ax, color="steelblue", edgecolor="none")
ax.set_title("Training label distribution — macro groups", fontsize=13)
ax.set_xlabel("")
ax.set_ylabel("Count")
ax.tick_params(axis="x", rotation=35)
for bar in ax.patches:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 40,
            f"{int(bar.get_height()):,}", ha="center", va="bottom", fontsize=8)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "label_distribution.png", bbox_inches="tight")
plt.show()

# the imbalance is real — sadness, joy, anger dominate; class_weight='balanced' will be needed in training
print(counts.to_string())

In [ ]:
train_df["n_chars"] = train_df["text"].str.len()
train_df["n_words"] = train_df["text"].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(train_df["n_chars"], bins=60, color="coral", edgecolor="none")
axes[0].set_title("Character length")
axes[0].set_xlabel("Characters")

axes[1].hist(train_df["n_words"], bins=40, color="mediumseagreen", edgecolor="none")
axes[1].set_title("Word count")
axes[1].set_xlabel("Words")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "text_length_dist.png", bbox_inches="tight")
plt.show()

print(train_df[["n_chars", "n_words"]].describe().round(1))

In [ ]:
SHOW = ["joy", "sadness", "anger", "love", "fear", "curiosity", "excitement", "gratitude", "optimism"]

fig, axes = plt.subplots(3, 3, figsize=(15, 10))
for ax, emotion in zip(axes.flat, SHOW):
    corpus = " ".join(train_df.loc[train_df["label"] == emotion, "text"])
    wc = WordCloud(
        width=400, height=250,
        background_color="white",
        colormap="viridis",
        max_words=80,
        collocations=False,
    ).generate(corpus)
    ax.imshow(wc, interpolation="bilinear")
    ax.set_title(emotion.capitalize(), fontsize=12, fontweight="bold")
    ax.axis("off")

plt.suptitle("Word clouds by macro-emotion (training set)", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "word_clouds.png", bbox_inches="tight")
plt.show()

In [ ]:
# co-occurrence heatmap — how often do two macro-emotions appear together in the same raw example?
comat = np.zeros((len(MACRO_LABELS), len(MACRO_LABELS)), dtype=int)
label_to_idx = {l: i for i, l in enumerate(MACRO_LABELS)}

for ex in ds["train"]:
    macros = list({MACRO_MAP[l] for l in ex["labels"]})
    for i in range(len(macros)):
        for j in range(i, len(macros)):
            a, b = label_to_idx[macros[i]], label_to_idx[macros[j]]
            comat[a][b] += 1
            if a != b:
                comat[b][a] += 1

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(comat, annot=True, fmt="d", xticklabels=MACRO_LABELS, yticklabels=MACRO_LABELS,
            cmap="Blues", linewidths=0.4, ax=ax)
ax.set_title("Macro-emotion co-occurrence in training examples", fontsize=12)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "cooccurrence_heatmap.png", bbox_inches="tight")
plt.show()

In [ ]:
# drop helper columns before saving
train_df.drop(columns=["n_chars", "n_words"], inplace=True)

train_df.to_csv(DATA_DIR / "processed" / "train.csv", index=False)
val_df.to_csv(DATA_DIR / "processed"   / "val.csv",   index=False)
test_df.to_csv(DATA_DIR / "processed"  / "test.csv",  index=False)

# also persist the macro map so downstream notebooks don't redefine it
import json
with open(DATA_DIR / "processed" / "macro_map.json", "w") as f:
    json.dump(MACRO_MAP, f, indent=2)

print("Saved processed splits and macro_map.json to data/processed/")